In [1]:
import torch

In [2]:
vocab_size = 4
emb_dim = 8
num_heads = 2
max_seq_len = 8
max_vertices = 16
layers = [(2, 2), 2, (1,1)]
out_layers = 2
pad_idx = 0

In [3]:
batch_size = 2
factor = 2

In [4]:
from model.model import EncoderDecoderDAG

In [5]:
model = EncoderDecoderDAG(vocab_size, emb_dim, num_heads, max_seq_len, max_vertices, layers, out_layers)

In [6]:
from utils.data import remove_padding_cols, process_data

In [7]:
enc_tokens = []
for i in range(batch_size):
    curr_len = torch.randint(1, max_seq_len, (1,)).item()
    enc_tokens.append(torch.randint(1, vocab_size, (curr_len,)))

In [8]:
enc_tokens = torch.nested.as_nested_tensor(enc_tokens)
enc_tokens = torch.nested.to_padded_tensor(enc_tokens, pad_idx, (batch_size, max_seq_len))

g:\Projects\Visual Studio Code\LMTests\lmtest\lib\site-packages\torch\nested\__init__.py:58: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ..\aten\src\ATen\NestedTensorImpl.cpp:180.)
  return torch._nested_tensor_from_tensor_list(tensor_list, dtype, None, device, None)


In [9]:
enc_tokens

tensor([[1, 1, 3, 2, 2, 0, 0, 0],
        [2, 3, 0, 0, 0, 0, 0, 0]])

In [10]:
enc_tokens = remove_padding_cols(enc_tokens, pad_idx)

In [18]:
_, l = enc_tokens.shape

In [19]:
decoder_tokens = torch.arange(0, l * factor).unsqueeze(0).expand(batch_size, -1)

In [20]:
decoder_tokens

tensor([[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
        [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]])

In [11]:
enc_tokens

tensor([[1, 1, 3, 2, 2],
        [2, 3, 0, 0, 0]])

In [12]:
token_lens, vertex_lens, token_mask, vertex_mask = process_data(enc_tokens, pad_idx=pad_idx, factor=2)

In [13]:
token_lens

tensor([5, 2])

In [14]:
vertex_lens

tensor([10,  4])

In [15]:
token_mask

tensor([[ True,  True,  True,  True,  True],
        [ True,  True, False, False, False]])

In [16]:
vertex_mask

tensor([[ True,  True,  True,  True,  True,  True,  True,  True,  True,  True],
        [ True,  True,  True,  True, False, False, False, False, False, False]])

In [22]:
log_transition_probs, log_emission_probs = model.forward(enc_tokens, decoder_tokens, token_mask, vertex_mask)

In [23]:
log_transition_probs.shape, log_emission_probs.shape

(torch.Size([2, 10, 10]), torch.Size([2, 10, 4]))